# Experiment on Generation
### locab generation API: python commmunication with Ollama

In [ ]:
import requests
import json

def test_ollama_connection(prompt_text):
    """
    Directly hits the local Ollama REST API to understand the data contract.
    """
    url = "http://localhost:11434/api/generate"
    
    payload = {
        "model": "llama3",
        "prompt": prompt_text,
        "stream": False # Set to False to get the whole response at once
    }
    
    response = requests.post(url, json=payload)
    
    if response.status_code == 200:
        return response.json()['response']
    else:
        return f"Error: {response.status_code} - {response.text}"

# Let's test basic reasoning without any RAG context.
question = "In one sentence, what is the core mechanism of the Transformer architecture?"
print("LLM Output:\n", test_ollama_connection(question))

## RAG prompt template

In [ ]:
# Let's simulate the output from your Retrieval & Re-ranking phase.
# We will use realistic excerpts from the famous "Attention Is All You Need" paper.

retrieved_chunks =[
    {
        "id": "chunk_44",
        "text": "The Transformer is the first transduction model relying entirely on self-attention to compute representations of its input and output without using sequence-aligned RNNs or convolution.",
        "metadata": {"source": "Attention Is All You Need, Vaswani et al. 2017", "paragraph": 2}
    },
    {
        "id": "chunk_89",
        "text": "Since our model contains no recurrence and no convolution, in order for the model to make use of the order of the sequence, we must inject some information about the relative or absolute position of the tokens. To this end, we add 'positional encodings' to the input embeddings.",
        "metadata": {"source": "Attention Is All You Need, Vaswani et al. 2017", "paragraph": 5}
    }
]

# Step 1: Format the chunks into a single readable context string
formatted_context = ""
for i, chunk in enumerate(retrieved_chunks, 1):
    formatted_context += f"[Doc {i}]\nSource: {chunk['metadata']['source']}\nContent: {chunk['text']}\n\n"

print("--- Formatted Context Sent to LLM ---")
print(formatted_context)

# Step 2: The CoT & Citation Prompt Template
prompt_template = f"""
You are an expert AI research assistant. Your task is to answer the user's question based strictly on the provided documents.

<Instructions>
1. You must base your answer ONLY on the provided context. If the answer is not in the context, say "I cannot answer this based on the provided documents."
2. Think step-by-step. First, analyze what the question is asking. Second, identify which documents contain the answer. Third, synthesize the answer.
3. CITATIONS: Every factual claim in your final answer MUST include an inline citation to the document it came from. Format citations as [Doc X].

<Context>
{formatted_context}

<Question>
Why does the Transformer need positional encodings, and how does it differ from RNNs?

<Answer>
"""

print("\n--- Testing the Prompt with Ollama ---")
print(test_ollama_connection(prompt_template))

### Testing Retriever pipeline flow into LLM generator

In [ ]:

# 1. Import your custom modules
import sys, os
PROJECT_ROOT = os.path.abspath('..')
sys.path.insert(0, PROJECT_ROOT)

from src.retrieval.hybrid_retriever import HybridRetriever
from src.retrieval.reranker import Reranker
from src.generation.llm_generator import LocalLLMGenerator

def test_full_rag_pipeline(user_query: str):
    print(f"User Query: {user_query}\n")
    
    # --- PHASE 3: RETRIEVAL ---
    print("1. Waking up the Brain (Retrieving & Reranking)...")
    # Initialize retriever pipeline components
    DENSE_INDEX = os.path.join(PROJECT_ROOT, "data/indices/dense.index")
    DENSE_META = os.path.join(PROJECT_ROOT, "data/indices/dense.index.meta")
    SPARSE_INDEX = os.path.join(PROJECT_ROOT, "data/indices/sparse.pkl")
    retriever = HybridRetriever(
        dense_index_path=DENSE_INDEX,
        dense_meta_path=DENSE_META,
        sparse_index_path=SPARSE_INDEX,
        embedding_model_name="BAAI/bge-small-en-v1.5"
    )
    reranker = Reranker(model_name="cross-encoder/ms-marco-MiniLM-L-6-v2")
    
    # The actual search (Stage 1: Broad search, Stage 2: Precision filtering)
    broad_results = retriever.search(user_query, k=50)
    top_5_docs = reranker.rerank(user_query, broad_results, top_k=5)
    print(f"-> Successfully retrieved {len(top_5_docs)} chunks from the NLP corpus.\n")

    # --- THE HANDOFF ---
    # At this exact moment, top_5_docs is passed from retriever Phase to the generation Phase in memory.

    # --- GENERATION ---
    print("2. Waking up the Mouth (LLM Generation)...")
    generator = LocalLLMGenerator(model_name="llama3")
    
    # The LLM reads the top 5 docs and generates the cited answer
    final_answer = generator.generate_answer(user_query, retrieved_docs=top_5_docs)
    
    return final_answer

# Test the pipeline with a real NLP question
query = "What is the vanishing gradient problem in Recurrent Neural Networks?"
response = test_full_rag_pipeline(query)

print("\n=== FINAL RAG RESPONSE ===")
print(response)


/home/nguegnang/anaconda3/envs/qasper-rag/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-03-01 02:46:11,757 - INFO - Use pytorch device_name: cpu
2026-03-01 02:46:11,758 - INFO - Load pretrained SentenceTransformer: BAAI/bge-small-en-v1.5
2026-03-01 02:46:11,935 - INFO - HTTP Request: HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
2026-03-01 02:46:11,951 - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-small-en-v1.5/5c38ec7c405ec4b44b94cc5a9bb96e735b38267a/modules.json "HTTP/1.1 200 OK"


User Query: What is the vanishing gradient problem in Recurrent Neural Networks?

1. Waking up the Brain (Retrieving & Reranking)...
Loading Embedding Model...


2026-03-01 02:46:12,077 - INFO - HTTP Request: HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"
2026-03-01 02:46:12,094 - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-small-en-v1.5/5c38ec7c405ec4b44b94cc5a9bb96e735b38267a/config_sentence_transformers.json "HTTP/1.1 200 OK"
2026-03-01 02:46:12,219 - INFO - HTTP Request: HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"
2026-03-01 02:46:12,220 - WARNING - Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
2026-03-01 02:46:12,234 - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-small-en-v1.5/5c38ec7c405ec4b44b94cc5a9bb96e735b38267a/config_sentence_transformers.json "HTTP/1.1 200 OK"
2026-03-01 02:46:12,352 - INFO - HTT

Loading FAISS Index from data/indices/dense.index...


RuntimeError: Error in faiss::FileIOReader::FileIOReader(const char*) at /project/third-party/faiss/faiss/impl/io.cpp:69: Error: 'f' failed: could not open data/indices/dense.index for reading: No such file or directory